# Energy-Distance Analysis

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from matplotlib.ticker import AutoMinorLocator, MaxNLocator
import os

def load_data_file(filepath, CONFIG):
    """Load and process molecular interaction data"""
    try:
        data = pd.read_csv(filepath, sep=r'\s+', header=None, comment='#')
        data.columns = ['phi1', 'phi2', 'zeta', 'distance', 'energy']
        data['energy_normalized'] = data['energy'] - 2*CONFIG['ref_energy']
        return data
    except:
        print(f"Error loading {filepath}")
        return None

def style_axis(ax, CONFIG, xlabel='Intermolecular distance (Å)', ylabel='Interaction energy (eV)'):
    """Apply consistent professional styling to axes"""
    ax.set_xlim(CONFIG['x_range'])
    ax.set_ylim(CONFIG['y_range'])
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.tick_params(axis='both', which='major', labelsize=10, width=1.0, length=4)
    ax.tick_params(axis='both', which='minor', width=0.5, length=2)
    ax.tick_params(direction='in', which='both')
    ax.tick_params(top=True, right=True, which='both')
    ax.xaxis.set_minor_locator(AutoMinorLocator(5))
    ax.yaxis.set_minor_locator(AutoMinorLocator(5))
    ax.xaxis.set_major_locator(MaxNLocator(6))
    ax.yaxis.set_major_locator(MaxNLocator(6))
    ax.grid(True, which='major', linewidth=0.3, color='#DDDDDD', alpha=0.7)
    ax.grid(True, which='minor', linewidth=0.2, color='#DDDDDD', alpha=0.4)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_linewidth(1.0)
        spine.set_color('#333333')
        
def plot_Ropt(interaction, highlight_configs):
    plt.rcParams.update({
        'font.size': 9, 'axes.labelsize': 10, 'xtick.labelsize': 8, 'ytick.labelsize': 8,
        'figure.dpi': 150, 'savefig.dpi': 600, 'text.usetex': False, 'mathtext.default': 'regular'
    })

    # --------------------------------------------------
    #       Configuration and utility functions
    # --------------------------------------------------

    INTERACTION_TYPE= interaction

    INTERACTION_CONFIGS = highlight_configs

    CONFIG = {
        'interaction_type': INTERACTION_TYPE,
        'ref_energy': -6227.1749,
        'figure_size': (3.5, 2.8),
        'x_range': (8, 11),
        'y_range': (-1.5, 0),
        'sampling_interval': 10,
        'colors': ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4'],
        'marker_styles': ['o', 's', '^', '*'],
        'configurations': INTERACTION_CONFIGS.get(INTERACTION_TYPE, [])
    }

    # Load data
    ropt_data = load_data_file(os.path.join(f"../E_Ropt_{INTERACTION_TYPE}.dat"), CONFIG)
    all_data = load_data_file(os.path.join(f"../E_all_{INTERACTION_TYPE}.dat"), CONFIG)

    print(f"Loaded data for {INTERACTION_TYPE} interaction:\n"
        f"R_opt ({len(ropt_data) if ropt_data is not None else 0} points),\n"
        f"All ({len(all_data) if all_data is not None else 0} points)")
    
    # --------------------------------------------------
    #       Figure 1: single interaction + configs
    # --------------------------------------------------
    
    fig, ax = plt.subplots(1, 1, figsize=CONFIG['figure_size'])

    if ropt_data is not None and all_data is not None:
        ropt_sampled = ropt_data.iloc[::CONFIG['sampling_interval']]
        ax.scatter(ropt_sampled['distance'], ropt_sampled['energy_normalized'], 
                c='#00000000', s=8, alpha=0.009, zorder=1, edgecolors='none')
        
        for i, config in enumerate(CONFIG['configurations']):
            phi1, phi2, zeta = config['phi1'], config['phi2'], config['zeta']
            mask = (all_data['phi1'] == phi1) & (all_data['phi2'] == phi2) & (np.abs(all_data['zeta'] - zeta) < 1e-2)
            config_data = all_data[mask].sort_values('distance')
            
            if len(config_data) > 0:
                label = f"φ={phi1}, χ={(phi1 - phi2) if CONFIG['interaction_type']=='SS' or CONFIG['interaction_type']=='SD' else (phi1 + phi2)}, ζ={zeta:.1f}"
                ax.plot(config_data['distance'], config_data['energy_normalized'],
                    color=CONFIG['colors'][i], linewidth=1.5, marker=CONFIG['marker_styles'][i], markersize=3,
                    label=label, zorder=2)

    style_axis(ax, CONFIG)

    legend = ax.legend(loc='lower right', frameon=True, fontsize=9, 
                    borderpad=0.3, handlelength=1.5)
    legend.get_frame().set_linewidth(0.5)
    legend.get_frame().set_edgecolor('#CCCCCC')
    legend.get_frame().set_facecolor('white')
    legend.get_frame().set_alpha(0.9)

    plt.tight_layout(pad=0.2)
    print("Figure 1: SS interaction configurations created")
    
    # --------------------------------------------------
    #       Figure 2: all interaction R opts
    # --------------------------------------------------
    
    fig2, ax2 = plt.subplots(1, 1, figsize=CONFIG['figure_size'])

    interaction_config = {
        'SS': {'file': 'E_Ropt_minmax_SS.dat', 'color': '#d62728', 'label': 'SS interactions'},
        'SD': {'file': 'E_Ropt_minmax_SD.dat', 'color': '#ff7f0e', 'label': 'SD interactions'},
        'DS': {'file': 'E_Ropt_minmax_DS.dat', 'color': '#2ca02c', 'label': 'DS interactions'},
        'DD': {'file': 'E_Ropt_minmax_DD.dat', 'color': '#1f77b4', 'label': 'DD interactions'}
    }

    for interaction_type, config in interaction_config.items():
        try:
            data = pd.read_csv(config['file'], sep=r'\s+', header=None, 
                            names=['distance', 'E_min', 'E_max'])
            data['E_min_norm'] = data['E_min'] - 2*CONFIG['ref_energy']
            data['E_max_norm'] = data['E_max'] - 2*CONFIG['ref_energy']
            
            ax2.fill_between(data['distance'], data['E_min_norm'], data['E_max_norm'], 
                            alpha=0.04, color=config['color'], linewidth=0)
            ax2.plot(data['distance'], data['E_min_norm'], 
                    color=config['color'], linewidth=1.5, alpha=0.8)
            ax2.plot(data['distance'], data['E_max_norm'], 
                    color=config['color'], linewidth=1.5, alpha=0.8, label=config['label'])
                    
        except FileNotFoundError:
            print(f"Warning: {config['file']} not found")

    ax2.set_xlim([7.8, 10.8])
    ax2.set_ylim(CONFIG['y_range'])
    style_axis(ax2, CONFIG)

    legend2 = ax2.legend(loc='lower right', frameon=True, fontsize=9, 
                        borderpad=0.3, handlelength=1.5)
    legend2.get_frame().set_linewidth(0.5)
    legend2.get_frame().set_edgecolor('#CCCCCC')
    legend2.get_frame().set_facecolor('white')
    legend2.get_frame().set_alpha(0.9)

    plt.tight_layout(pad=0.2)
    print("Figure 2: Energy range comparison created")
    plt.show()

In [ ]:
interaction = 'DD'
highlighted_configs = {
    'SS': [
        {'phi1': 196, 'phi2': 16, 'zeta': 0.000000},
        {'phi1': 0, 'phi2': 0, 'zeta': 1.210625},
        {'phi1': 9, 'phi2': 5, 'zeta': 0.55875},
        {'phi1': 0, 'phi2': 0, 'zeta': 0.000000}
    ],
    'SD': [
        {'phi1': 100, 'phi2': 0, 'zeta': 1.303750},
        {'phi1': 0, 'phi2': 18, 'zeta': 0.745000},
        {'phi1': 270, 'phi2': 11, 'zeta': 1.024375},
        {'phi1': 275, 'phi2': 0, 'zeta': 0.279375}
    ],
    'DS': [
        {'phi1': 70, 'phi2': 10, 'zeta': 1.117500},
        {'phi1': 40, 'phi2': 15, 'zeta': 1.117500},
        {'phi1': 130, 'phi2': 10, 'zeta': 0.279375},
        {'phi1': 7, 'phi2': 15, 'zeta': 0.745000}
    ],
    'DD': [
        {'phi1': 69, 'phi2': 18, 'zeta': 1.396875},
        {'phi1': 0, 'phi2': 0, 'zeta': 0.000000},
        {'phi1': 22, 'phi2': 19, 'zeta': 0.465625},
        {'phi1': 305, 'phi2': 1, 'zeta': 1.396875}
    ]
}

plot_Ropt(interaction, highlighted_configs)